# PHT3D Example 13 — MF6PQC

实线为 MF6PQC 出口结果，黑色虚线为 PHT3D 出口结果，散点为案例附带的 Appelo 等实验数据。

MF6PQC 数据及计算时刻读取自 `output`。`input_data/PHT3D_13_results.npy` 保存从 PHT3D UCN 文件提取的出口浓度及各记录的实际时间，包含初始时刻和应力期交界时刻。`input_data/pht3d_reference` 中的 `phinp.dat` 与 `pht3d_ph.dat` 保存对应的 PHT3D 化学输入。

保留原图的面板布局、配色、线型、单位和坐标范围。HCO₃⁻ 面板沿用原案例的 C(4) 输出变量。

In [ ]:
import sys
from pathlib import Path

sys.dont_write_bytecode = True
CASE_NAME = "ex013_PHT3D_13"
candidates = (
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / CASE_NAME)
)
CASE_DIR = next(
    (
        path.resolve()
        for path in candidates
        if path.name == CASE_NAME and (path / "modflow_model.py").is_file()
    ),
    None,
)
if CASE_DIR is None:
    raise FileNotFoundError(f"Cannot locate examples/{CASE_NAME} from {Path.cwd()}")
EXAMPLES_DIR = CASE_DIR.parent
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from example_utils import load_results, read_headings, restore_archives, runtime_path, time_indices

CASE_FILE = CASE_DIR / "run.py"
INPUT_DIR = CASE_DIR / "input_data"
OUTPUT_DIR = runtime_path(CASE_FILE, "output")
SIMULATION_DIR = runtime_path(CASE_FILE, "simulation")
restore_archives(OUTPUT_DIR)
restore_archives(SIMULATION_DIR)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

FLOW_RATE = 2.4e-4


def output_times() -> np.ndarray:
    return np.load(runtime_path(CASE_FILE, "output") / "results_times.npy", allow_pickle=False)


def load_results(case_dir: Path) -> tuple[np.ndarray, list[str], np.ndarray]:
    results, headings, result_times = load_results(runtime_path(CASE_FILE, "output"))

    observations = np.loadtxt(INPUT_DIR / "observations.txt")
    if results.shape != (len(output_times()), len(headings), 16):
        raise ValueError("The result array, timestamps and headings have inconsistent dimensions")
    return results, headings, observations


def load_official_reference(case_dir: Path) -> dict[str, np.ndarray]:
    archive, headings, result_times = load_results(INPUT_DIR)
    reference = {name: archive[name] for name in archive.dtype.names}
    times = reference["time_days"]
    if np.any(np.diff(times) <= 0) or any(
        reference[name].shape != times.shape for name in ("Cl", "S_6", "Mg", "C_4", "pH", "Ca")
    ):
        raise ValueError("The PHT3D concentration records and timestamps are inconsistent")
    return reference


def plot_breakthrough(case_dir: Path) -> tuple[plt.Figure, np.ndarray]:
    results, headings, observations = load_results(case_dir)
    reference = load_official_reference(case_dir)
    volume_ml = output_times() * FLOW_RATE * 1.0e6
    reference_volume_ml = reference["time_days"] * FLOW_RATE * 1.0e6
    outlet = results[:, :, -1]
    panels = (
        ("Cl", "Cl", 4, "blue", (0, 0.025)),
        (r"SO$_4^{2-}$", "S_6", 5, "blue", (0, 0.010)),
        ("Mg", "Mg", 1, "red", (0, 0.025)),
        (r"HCO$_3^-$", "C_4", 3, "red", (0, 0.015)),
        ("pH", "pH", 6, "green", (4, 10)),
        (r"Ca$^{2+}$", "Ca", 2, "green", (0, 0.006)),
    )

    fig, axes = plt.subplots(3, 2, figsize=(8.0, 7.0), sharex=True)
    for axis, (title, heading, obs_col, color, ylim) in zip(axes.flat, panels, strict=False):
        model = outlet[:, headings.index(heading)]
        valid = observations[:, obs_col] > -9999
        axis.plot(
            observations[valid, 0],
            observations[valid, obs_col],
            ".",
            color=color,
            markersize=5,
            label="Appelo et al.",
        )
        axis.plot(
            volume_ml,
            model,
            "-",
            color=color,
            linewidth=1.5,
            label="MF6PQC",
        )
        axis.plot(
            reference_volume_ml,
            reference[heading],
            "--",
            color="black",
            linewidth=1.0,
            label="PHT3D",
        )
        axis.set_title(title, pad=2)
        axis.set_xlim(-100, 800)
        axis.set_ylim(*ylim)
        axis.set_ylabel("mol/L" if heading != "pH" else "")
        axis.tick_params(direction="in", top=True, right=True)
    axes[2, 0].set_xlabel("ml outflow")
    axes[2, 1].set_xlabel("ml outflow")
    axes[0, 0].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    return fig, axes


In [ ]:
plt.rcParams.update({"font.size": 9, "figure.dpi": 120})


In [ ]:
fig, axes = plot_breakthrough(CASE_DIR)
plt.show()
